# 🤖 AI-Powered Customer Churn Intelligence System
### End-to-End ML Pipeline: Prediction → Explainability → Business Recommendations
---
**Author:** lrcataytay00745-spec  
**Dataset:** IBM Telco Customer Churn  
**Tools:** Python, Pandas, Scikit-learn, XGBoost, SHAP, Matplotlib, Seaborn

## STEP 1 — Install & Import Libraries

In [ ]:
!pip install xgboost shap -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

from xgboost import XGBClassifier
import shap

import warnings
warnings.filterwarnings('ignore')

print('✅ All libraries loaded successfully')

## STEP 2 — Load Data

In [ ]:
# Upload your CSV to Colab as /content/customer_churn.csv
df = pd.read_csv('/content/customer_churn.csv')

print(f'Dataset shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
df.head()

In [ ]:
# Data inspection
print('=== DATA TYPES ===')
print(df.dtypes)
print('\n=== MISSING VALUES ===')
print(df.isnull().sum())
print('\n=== TARGET DISTRIBUTION ===')
print(df['Churn'].value_counts())
print(f'Churn Rate: {df["Churn"].value_counts(normalize=True)["Yes"]:.1%}')

## STEP 3 — Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Churn distribution
df['Churn'].value_counts().plot(kind='bar', ax=axes[0], color=['#2ecc71','#e74c3c'])
axes[0].set_title('Churn Distribution')
axes[0].set_xlabel('Churn')
axes[0].set_ylabel('Count')
axes[0].tick_params(rotation=0)

# Monthly Charges by Churn
df.boxplot(column='MonthlyCharges', by='Churn', ax=axes[1])
axes[1].set_title('Monthly Charges by Churn')

# Tenure by Churn
df.boxplot(column='tenure', by='Churn', ax=axes[2])
axes[2].set_title('Tenure by Churn')

plt.tight_layout()
plt.savefig('eda_overview.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ EDA charts saved')

## STEP 4 — Data Cleaning

In [ ]:
# Remove customerID (not a predictive feature)
df.drop('customerID', axis=1, inplace=True)

# Convert TotalCharges to numeric (sometimes read as string)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# Fill missing TotalCharges with 0 (new customers)
df['TotalCharges'].fillna(0, inplace=True)

# Encode target variable
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

print(f'Cleaned dataset shape: {df.shape}')
print('✅ Data cleaning complete')

## STEP 5 — Feature Engineering

In [ ]:
# Feature 1: Average Monthly Spend (more stable than TotalCharges alone)
df['AvgMonthlySpend'] = df['TotalCharges'] / (df['tenure'] + 1)

# Feature 2: High Value Customer flag
df['HighValue'] = np.where(df['MonthlyCharges'] > 80, 1, 0)

# Feature 3: Long-term Customer flag (>2 years)
df['LongTerm'] = np.where(df['tenure'] > 24, 1, 0)

# Feature 4: Early Churn Risk (honeymoon period)
df['EarlyRisk'] = np.where(df['tenure'] < 6, 1, 0)

# Feature 5: Charge Growth Rate (proxy for price sensitivity)
df['ChargeGrowthRate'] = np.where(
    df['tenure'] > 0,
    (df['MonthlyCharges'] - df['AvgMonthlySpend']) / (df['AvgMonthlySpend'] + 1),
    0
)

print(f'Features after engineering: {df.shape[1]}')
print('New features: AvgMonthlySpend, HighValue, LongTerm, EarlyRisk, ChargeGrowthRate')
print('✅ Feature engineering complete')

## STEP 6 — Encode Categoricals & Train-Test Split

In [ ]:
# One-hot encode all categorical variables
df_encoded = pd.get_dummies(df, drop_first=True)

print(f'Encoded dataset shape: {df_encoded.shape}')

# Split features and target
X = df_encoded.drop('Churn', axis=1)
y = df_encoded['Churn']

# Train-test split (80/20, stratified to preserve class balance)
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f'Training set: {X_train.shape[0]} rows')
print(f'Test set: {X_test.shape[0]} rows')
print('✅ Split complete')

## STEP 7 — Baseline Model: Logistic Regression

In [ ]:
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train, y_train)

y_pred_lr = lr.predict(X_test)
probs_lr = lr.predict_proba(X_test)[:, 1]

print('=== LOGISTIC REGRESSION ===')
print(classification_report(y_test, y_pred_lr))
print(f'ROC-AUC: {roc_auc_score(y_test, probs_lr):.4f}')

## STEP 8 — Advanced Models: Random Forest & XGBoost

In [ ]:
# Random Forest
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)
probs_rf = rf.predict_proba(X_test)[:, 1]

print('=== RANDOM FOREST ===')
print(classification_report(y_test, y_pred_rf))
print(f'ROC-AUC: {roc_auc_score(y_test, probs_rf):.4f}')

In [ ]:
# XGBoost — Primary Model
xgb = XGBClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=5,
    use_label_encoder=False,
    eval_metric='logloss',
    random_state=42
)
xgb.fit(X_train, y_train)

y_pred_xgb = xgb.predict(X_test)
probs_xgb = xgb.predict_proba(X_test)[:, 1]

print('=== XGBOOST ===')
print(classification_report(y_test, y_pred_xgb))
print(f'ROC-AUC: {roc_auc_score(y_test, probs_xgb):.4f}')

## STEP 9 — Model Comparison Chart

In [ ]:
from sklearn.metrics import roc_curve

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# ROC Curves
for name, probs in [('Logistic Regression', probs_lr), ('Random Forest', probs_rf), ('XGBoost', probs_xgb)]:
    fpr, tpr, _ = roc_curve(y_test, probs)
    auc = roc_auc_score(y_test, probs)
    axes[0].plot(fpr, tpr, label=f'{name} (AUC={auc:.3f})')
axes[0].plot([0, 1], [0, 1], 'k--', label='Random Baseline')
axes[0].set_title('ROC Curves — Model Comparison')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].legend()
axes[0].grid(True)

# Model AUC Bar Chart
models = ['Logistic\nRegression', 'Random\nForest', 'XGBoost']
aucs = [roc_auc_score(y_test, probs_lr), roc_auc_score(y_test, probs_rf), roc_auc_score(y_test, probs_xgb)]
bars = axes[1].bar(models, aucs, color=['#3498db', '#2ecc71', '#e74c3c'])
axes[1].set_ylim(0.7, 1.0)
axes[1].set_title('ROC-AUC Score by Model')
axes[1].set_ylabel('ROC-AUC Score')
for bar, auc in zip(bars, aucs):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005, f'{auc:.4f}', ha='center')
axes[1].grid(axis='y')

plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Model comparison chart saved')

## STEP 10 — Churn Risk Scoring & Segmentation

In [ ]:
# Create risk score dataset from test set
df_test = X_test.copy()
df_test['ChurnRisk'] = probs_xgb
df_test['PredictedChurn'] = y_pred_xgb
df_test['ActualChurn'] = y_test.values

# Risk segmentation function
def risk_level(x):
    if x > 0.75:
        return 'High Risk'
    elif x > 0.40:
        return 'Medium Risk'
    else:
        return 'Low Risk'

df_test['RiskSegment'] = df_test['ChurnRisk'].apply(risk_level)

print('=== RISK SEGMENT DISTRIBUTION ===')
print(df_test['RiskSegment'].value_counts())
print('\n=== CHURN RATE BY SEGMENT ===')
print(df_test.groupby('RiskSegment')['ActualChurn'].mean().sort_values(ascending=False).map('{:.1%}'.format))

## STEP 11 — SHAP Explainability

In [ ]:
# Initialize SHAP explainer
explainer = shap.Explainer(xgb)
shap_values = explainer(X_test)

# SHAP Summary Plot — Global Feature Importance
plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X_test, show=False)
plt.title('SHAP Feature Importance — Churn Drivers', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('shap_summary.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ SHAP summary plot saved')

In [ ]:
# SHAP Bar Plot — Mean absolute SHAP values
plt.figure(figsize=(10, 6))
shap.plots.bar(shap_values, show=False)
plt.title('Top Churn Drivers (Mean |SHAP Value|)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('shap_bar.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ SHAP bar chart saved')

In [ ]:
# Waterfall plot — Individual prediction explanation (first high-risk customer)
high_risk_idx = df_test[df_test['RiskSegment'] == 'High Risk'].index[0]
position = list(X_test.index).index(high_risk_idx)

plt.figure(figsize=(12, 6))
shap.plots.waterfall(shap_values[position], show=False)
plt.title('Individual Churn Prediction Explanation — High Risk Customer', fontsize=12)
plt.tight_layout()
plt.savefig('shap_waterfall.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Waterfall explanation chart saved')

## STEP 12 — Business KPIs: Revenue at Risk

In [ ]:
# Add MonthlyCharges back if available
# Revenue at Risk = MonthlyCharges × ChurnProbability
if 'MonthlyCharges' in df_test.columns:
    df_test['RevenueAtRisk'] = df_test['MonthlyCharges'] * df_test['ChurnRisk']
else:
    # Use AvgMonthlySpend as proxy
    charge_col = [c for c in df_test.columns if 'charge' in c.lower() or 'spend' in c.lower()][0]
    df_test['RevenueAtRisk'] = df_test[charge_col] * df_test['ChurnRisk']

total_revenue_at_risk = df_test['RevenueAtRisk'].sum()
high_risk_count = (df_test['RiskSegment'] == 'High Risk').sum()
churn_rate = df_test['ActualChurn'].mean()

print('=== EXECUTIVE KPI SUMMARY ===')
print(f'Total Customers Analyzed:   {len(df_test):,}')
print(f'Overall Churn Rate:          {churn_rate:.1%}')
print(f'High-Risk Customers:         {high_risk_count:,}')
print(f'Total Revenue at Risk/Month: ${total_revenue_at_risk:,.2f}')
print(f'Avg Revenue at Risk/Customer: ${total_revenue_at_risk/len(df_test):,.2f}')

## STEP 13 — Export for Power BI Dashboard

In [ ]:
# Build final dashboard export dataset
final_df = df_test.copy()

# Add retention recommendation column
def recommend_action(row):
    if row['RiskSegment'] == 'High Risk':
        if 'MonthlyCharges' in row and row.get('MonthlyCharges', 0) > 80:
            return 'Offer Loyalty Discount'
        elif row.get('EarlyRisk', 0) == 1:
            return 'Enhanced Onboarding Program'
        else:
            return 'Priority Retention Outreach'
    elif row['RiskSegment'] == 'Medium Risk':
        return 'Proactive Check-in & Survey'
    else:
        return 'Standard Engagement'

final_df['RetentionAction'] = final_df.apply(recommend_action, axis=1)

# Export to CSV
final_df.to_csv('churn_dashboard_data.csv', index=False)

print(f'✅ Exported {len(final_df):,} rows to churn_dashboard_data.csv')
print(f'Columns: {list(final_df.columns[:10])} ...')
print('\n👉 Download this file from Colab Files panel → Load into Power BI')

## ✅ Pipeline Complete!

**Files generated:**
- `churn_dashboard_data.csv` — Load into Power BI
- `eda_overview.png` — EDA charts
- `model_comparison.png` — ROC curves
- `shap_summary.png` — Feature importance
- `shap_bar.png` — Top churn drivers
- `shap_waterfall.png` — Individual explanation

**Next steps:**
1. Download `churn_dashboard_data.csv`
2. Open Power BI Desktop
3. Load CSV → Build 4-page dashboard
4. Screenshot dashboard pages
5. Upload everything to GitHub portfolio